# yolo_lpr_cpp — 認識器の学習 (M5) on Colab

無料枠の T4 で回す想定。手元 (CPU) は 4-5 サンプル/秒しか出ないので、本番の学習はここでやる。

この notebook がやること:
1. リポジトリを clone し、フォントを取得して C++ 生成器をビルド
2. 合成データを生成（OpenMP 並列。10 万枚で ~10 分）
3. 実データ (dyama/alpr_jp, MIT) を clone — 地域名ラベル付き 720 枚
4. 既存 ONNX 重みから学習を開始し、実データ hold-out で測りながら回す
5. ONNX を書き出して手元に持ち帰る（C++/WASM がそのまま読める形）

**測って決めたレシピ**（詳細は RESUME.md）:
- region head は**実データだけ**で学習（合成の字形は転移しない: 数字 72-92% に対し region 28%）
- BatchNorm の統計は**凍結**（合成が多いバッチで動かすと実データ精度が落ちる）
- backbone は lr×0.1、新しい head だけ lr

In [ ]:
!nvidia-smi -L
!git clone -q https://github.com/yomei-o/yolo_lpr_cpp.git
%cd yolo_lpr_cpp
!pip -q install onnx onnxruntime
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# フォント: Colab には Windows のシステムフォントが無いので OFL の GenSenRounded2-B だけになる。
# 生成のパリティ（C++ と Python が同じフォント集合を見ること）はこの環境の中では保たれる。
!python tools/fetch_fonts.py
!apt-get -qq install -y fonts-noto-cjk > /dev/null && cp /usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc fonts/ 2>/dev/null || true
!python tools/fetch_fonts.py --list

In [ ]:
# C++ 生成器をビルド（OpenMP 並列）。Python 側の生成器と同じラベルを出すことは
# tools/parity/gen.py で確認できる。
!g++ -std=c++20 -O2 -fopenmp -Ipure -Ipure/third_party pure/jlpr.cpp -o jlpr
!python tools/parity/gen.py --gen ./jlpr --count 200 --images 4

In [ ]:
# 合成データ（認識用 crop）。枚数はここで決める。10 万枚 = ディスク ~2GB。
N = 60000
!./jlpr gen --out data/synth --count $N --seed 90210 --quiet
!ls data/synth | head -3 && wc -l data/synth/labels.txt

In [ ]:
# 実データ（地域名ラベル付き 720 枚 + ネガ）。MIT、撮影者に著作権。
!git clone -q --depth 1 https://github.com/dyama/alpr_jp.git ../alpr_jp
!ls ../alpr_jp

In [ ]:
# 学習。step 0 の数字が「今出荷しているモデル」の実データ hold-out 精度で、そこを超えるのが目標。
!python tools/train_ocr.py \
  --synth data/synth --alpr ../alpr_jp \
  --steps 4000 --batch 64 --lr 4e-4 --real-weight 0.4 \
  --eval-every 500 --eval-limit 144 \
  --save ocr_v2.pt --export models/plate_ocr_v2.onnx

In [ ]:
# 出力 ONNX が本当に C++ 側の推論経路で動くか（op 集合が同じか）を確かめてから持ち帰る。
!python tools/eval_ocr.py --data ../alpr_jp --models ours --ours models/plate_ocr_v2.onnx --limit 200 --single
!python tools/eval_ocr.py --data data/synth --kind synth --models ours --ours models/plate_ocr_v2.onnx --limit 200 --single

In [ ]:
from google.colab import files
files.download('models/plate_ocr_v2.onnx')